# 🏥 Medical Report Analysis — Kaggle P100 Training Notebook

Fine-tunes **BioMistral-7B** on the full medical dataset using QLoRA on Kaggle's **free P100 GPU**.

### ⚙️ Before Running:
1. Make sure the **GPU accelerator is ON**: Go to `Settings (⚙️) → Accelerator → GPU P100`
2. Make sure **Internet is ON**: Go to `Settings (⚙️) → Internet → ON`
3. Run all cells from top to bottom. Training takes **~6-7 hours** on P100.
4. Outputs are saved to `/kaggle/working/lora_adapters/` automatically.

## 🔍 Step 0: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📦 Step 1: Clone Repo & Install Dependencies

In [ ]:
# Clone your GitHub repository
# Replace YOUR_TOKEN with your GitHub personal access token
!git clone https://YOUR_TOKEN@github.com/URVISH611711/report-analysis.git
%cd report-analysis

In [ ]:
# Install exact versions that match our fixed training script
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    peft==0.12.0 \
    trl==0.9.6 \
    accelerate==0.33.0 \
    bitsandbytes==0.43.3 \
    huggingface_hub \
    scipy \
    sentencepiece \
    rich

print("\n[OK] All dependencies installed!")

## 📥 Step 2: Download & Prepare Training Datasets

In [ ]:
from datasets import load_dataset, Dataset
from pathlib import Path
import json

OUTPUT_DATASET_DIR = Path("/kaggle/working/processed_dataset")
OUTPUT_DATASET_DIR.mkdir(parents=True, exist_ok=True)

def format_instruction(instruction, input_val, response):
    prompt = (
        f"<|im_start|>system\nYou are a clinical report analysis assistant. "
        f"Answer the patient's questions and analyze reports accurately.<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\nInput Data:\n{input_val}<|im_end|>\n"
        f"<|im_start|>assistant\n{response}<|im_end|>"
    )
    return {"text": prompt}

all_records = []

# 1. PubMedQA
print("[1/4] Downloading PubMedQA...")
try:
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled")
    for row in ds["train"]:
        context = " ".join(row["context"]["contexts"])
        all_records.append(format_instruction(
            f"Read this medical context and answer: {row['question']}",
            context[:800], row["long_answer"]
        ))
    print(f"   [OK] {len(ds['train']):,} samples")
except Exception as e:
    print(f"   [WARN] {e}")

# 2. MedQA USMLE
print("[2/4] Downloading MedQA-USMLE...")
try:
    ds = load_dataset("GBaker/MedQA-USMLE-4-options")
    for split in ["train", "test"]:
        if split in ds:
            for row in ds[split]:
                options = "\n".join([f"{k}: {v}" for k, v in row["options"].items()])
                all_records.append(format_instruction(
                    f"Solve this USMLE question:\n{row['question']}",
                    options, f"The correct answer is {row['answer_idx']}: {row['answer']}"
                ))
    total = sum(len(ds[s]) for s in ds)
    print(f"   [OK] {total:,} samples")
except Exception as e:
    print(f"   [WARN] {e}")

# 3. ChatDoctor (first 5000)
print("[3/4] Downloading ChatDoctor...")
try:
    ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
    subset = ds["train"].select(range(min(5000, len(ds["train"]))))
    for row in subset:
        all_records.append(format_instruction(
            row["input"] or "Medical Inquiry",
            row["instruction"], row["output"]
        ))
    print(f"   [OK] {len(subset):,} samples")
except Exception as e:
    print(f"   [WARN] {e}")

# 4. MedMCQA
print("[4/4] Downloading MedMCQA...")
try:
    ds = load_dataset("openlifescienceai/medmcqa")
    options_map = {0: "a", 1: "b", 2: "c", 3: "d"}
    for row in ds["train"].select(range(min(5000, len(ds["train"])))):
        options = f"A: {row['opa']}\nB: {row['opb']}\nC: {row['opc']}\nD: {row['opd']}"
        correct = ["A","B","C","D"][row["cop"]]
        all_records.append(format_instruction(
            f"Answer this medical exam question:\n{row['question']}",
            options, f"The correct answer is {correct}. {row.get('exp', '')}"
        ))
    print(f"   [OK] 5,000 samples")
except Exception as e:
    print(f"   [WARN] {e}")

print(f"\n[+] Total compiled: {len(all_records):,} training samples")

# Save as HuggingFace dataset with train/test split
hf_dataset = Dataset.from_list(all_records)
split_dataset = hf_dataset.train_test_split(test_size=0.05, seed=42)
split_dataset.save_to_disk(str(OUTPUT_DATASET_DIR))
print(f"[OK] Dataset saved: {len(split_dataset['train']):,} train | {len(split_dataset['test']):,} eval")

## 🧠 Step 3: QLoRA Fine-Tuning on Full Dataset
This is the main training cell. It will take approximately **5-7 hours** on a P100 GPU.

In [ ]:
import torch
from pathlib import Path
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

BASE_MODEL_ID   = "BioMistral/BioMistral-7B"
DATASET_DIR     = Path("/kaggle/working/processed_dataset")
OUTPUT_DIR      = Path("/kaggle/working/lora_adapters")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load dataset
print("[+] Loading processed dataset...")
dataset = load_from_disk(str(DATASET_DIR))
train_dataset = dataset["train"]
eval_dataset  = dataset["test"]
print(f"[+] Train: {len(train_dataset):,} | Eval: {len(eval_dataset):,}")

# Tokenizer
print(f"[+] Loading tokenizer for {BASE_MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 4-bit QLoRA config
print("[+] Configuring 4-bit BitsAndBytes quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load model
print("[+] Loading BioMistral-7B (this takes ~3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()

# LoRA config
print("[+] Configuring LoRA (r=16, alpha=32)...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Training config — Optimized for P100 16GB
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=True,                      # P100 uses fp16 (not bf16)
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    lr_scheduler_type="cosine",
    report_to="none",
    optim="paged_adamw_8bit",
    dataset_text_field="text",
    max_seq_length=1024,
)

# Initialize Trainer
print("[+] Initializing SFT Trainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_config,
    tokenizer=tokenizer,
    args=training_args,
)

# START TRAINING
print("\n" + "="*60)
print("  [+] Starting Full Training on Kaggle P100 GPU!")
print("  Expected time: ~5-7 hours for full dataset")
print("="*60)
trainer.train()

# Save final adapters
print(f"\n[SUCCESS] Training complete! Saving adapters to: {OUTPUT_DIR}")
trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
print("[OK] LoRA adapters saved to /kaggle/working/lora_adapters/")

## 💾 Step 4: Merge Adapters & Save Final Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL_ID  = "BioMistral/BioMistral-7B"
ADAPTERS_DIR   = Path("/kaggle/working/lora_adapters")
MERGED_DIR     = Path("/kaggle/working/merged_model")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print("[+] Loading base model in FP16 for merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print(f"[+] Loading LoRA adapters from: {ADAPTERS_DIR}")
peft_model = PeftModel.from_pretrained(base_model, str(ADAPTERS_DIR))

print("[+] Merging adapters into base model...")
merged_model = peft_model.merge_and_unload()

print(f"[+] Saving merged model to: {MERGED_DIR}")
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))

print("\n[SUCCESS] Final merged model saved to /kaggle/working/merged_model/")
print("You can now download it from the Kaggle Output tab!")